<a href="https://colab.research.google.com/github/bosslgtmte/MILAN-CSCC102-FileHandling/blob/main/MIlan_Lab9_File_Handling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Without GUI


In [5]:
import pandas as pd
from google.colab import files

print("--- Step 1: Upload Your CSV File ---")
uploaded = files.upload()

# Automatically grab the uploaded filename
filename = list(uploaded.keys())[0]
print(f"\nSuccessfully loaded: {filename}\n")

# Load the CSV into pandas
df = pd.read_csv(filename)

# Clean up the student name column by forward-filling missing rows
student_col = df.columns[0]
df[student_col] = df[student_col].ffill().fillna(student_col)

# Extract unique student names
student_list = df[student_col].unique()

print("========================================")
print("       STUDENT GRADE VIEWER             ")
print("========================================")
print("Available Students in Uploaded File:")
for i, name in enumerate(student_list, start=1):
    print(f"[{i}] {name}")
print("-" * 40)

# Interactive prompt to choose a student
search_query = input("Type a student's name (or part of it) to display: ").strip()

# Filter and display records for the matching student
matched_students = [name for name in student_list if search_query.lower() in name.lower()]

if matched_students:
    target_student = matched_students[0]
    student_records = df[df[student_col] == target_student]

    print(f"\n========================================")
    print(f" Grades for: {target_student.strip()}")
    print(f"========================================")

    # Clean, perfectly aligned custom table layout
    print(f"{'Course Code':<12} | {'Course Title':<55} | {'Units':<5} | {'Instructor':<25} | {'Grade':<6}")
    print("-" * 115)

    for _, row in student_records.iterrows():
        code = str(row['Course Code']).strip()
        title = str(row['Course Title']).strip()
        units = str(row['Units']).strip()
        instructor = str(row['Instructor']).strip()
        grade = str(row['Grade ']).strip()

        print(f"{code:<12} | {title:<55} | {units:<5} | {instructor:<25} | {grade:<6}")
else:
    print(f"\n❌ No student matching '{search_query}' was found in the dataset.")

--- Step 1: Upload Your CSV File ---


Saving course_lists.csv to course_lists (4).csv

Successfully loaded: course_lists (4).csv

       STUDENT GRADE VIEWER             
Available Students in Uploaded File:
[1] Gabriel M. Villanueva
[2] Mateo Miguel De Guzman
[3] Hannah Nicole Roxas
[4] Joshua Paolo Navarro
[5] Andrea Isabel Mercado
[6] Rafael Carlos Lorenzo
[7] John David Mendoza
[8] Samantha Alyssa Cruz
[9] Christian James Soriano
[10] Denver Chris Milan 
----------------------------------------
Type a student's name (or part of it) to display: Denver

 Grades for: Denver Chris Milan
Course Code  | Course Title                                            | Units | Instructor                | Grade 
-------------------------------------------------------------------------------------------------------------------
CSCC 101     | Introduction to Computing                               | 3     | Ma'am Cari-An             | 1.5   
MST 111      | Mathematics in the Modern World                         | 3     | Ma'am Gallaron 

# With GUI


In [10]:
import io
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

# Global variable to hold our loaded DataFrame
df_global = None

# UI Elements with enhanced styling
upload_btn = widgets.Button(description="Upload CSV File", button_style='primary', icon='upload', layout=widgets.Layout(width='180px', margin='10px'))
student_dropdown = widgets.Dropdown(description="Student:", options=[], disabled=True, layout=widgets.Layout(width='300px', margin='10px'))
view_btn = widgets.Button(description="View Grades", button_style='success', icon='table', disabled=True, layout=widgets.Layout(width='180px', margin='10px'))
output_area = widgets.Output()

def clean_grade(val):
    try:
        return float(str(val).strip())
    except ValueError:
        return None

def on_upload_clicked(b):
    global df_global
    with output_area:
        clear_output()
        print("Please select your CSV file...")
        uploaded = files.upload()
        if not uploaded:
            print("Upload cancelled.")
            return

        try:
            filename = list(uploaded.keys())[0]
            df = pd.read_csv(io.BytesIO(uploaded[filename]))

            # Clean up the student name column by forward-filling missing rows
            student_col = df.columns[0]
            df[student_col] = df[student_col].ffill().fillna(student_col)

            df_global = df

            # Populate dropdown
            student_list = sorted(list(df[student_col].unique()))
            student_dropdown.options = student_list
            student_dropdown.disabled = False
            view_btn.disabled = False

            display(widgets.HTML(f"<div style='color: #2ecc71; font-weight: bold; margin-top: 10px;'>✅ Loaded successfully: {filename}</div>"))
        except Exception as e:
            display(widgets.HTML(f"<div style='color: #e74c3c; font-weight: bold; margin-top: 10px;'>❌ Failed to load file: {e}</div>"))

def on_view_clicked(b):
    global df_global
    with output_area:
        clear_output()
        if df_global is None:
            print("Please upload a CSV file first!")
            return

        selected_student = student_dropdown.value
        student_col = df_global.columns[0]
        filtered_df = df_global[df_global[student_col] == selected_student].copy()

        # Standardize columns
        grade_col = 'Grade ' if 'Grade ' in filtered_df.columns else 'Grade'
        filtered_df['NumericGrade'] = filtered_df[grade_col].apply(clean_grade)
        filtered_df['Units'] = pd.to_numeric(filtered_df['Units'], errors='coerce').fillna(0)

        # Calculate Metrics
        valid_grades = filtered_df['NumericGrade'].dropna()
        avg_grade = valid_grades.mean() if not valid_grades.empty else 0.0
        total_units = filtered_df['Units'].sum()

        # --- Graphical Elements: KPI Cards ---
        kpi_html = f"""
        <div style="display: flex; gap: 20px; margin-bottom: 20px;">
            <div style="background-color: #2c3e50; color: white; padding: 15px 25px; border-radius: 8px; flex: 1; text-align: center; box-shadow: 2px 2px 5px rgba(0,0,0,0.1);">
                <div style="font-size: 14px; text-transform: uppercase; opacity: 0.8;">Selected Student</div>
                <div style="font-size: 20px; font-weight: bold; margin-top: 5px;">{selected_student}</div>
            </div>
            <div style="background-color: #3498db; color: white; padding: 15px 25px; border-radius: 8px; width: 180px; text-align: center; box-shadow: 2px 2px 5px rgba(0,0,0,0.1);">
                <div style="font-size: 14px; text-transform: uppercase; opacity: 0.8;">Average Grade</div>
                <div style="font-size: 24px; font-weight: bold; margin-top: 5px;">{avg_grade:.2f}</div>
            </div>
            <div style="background-color: #e67e22; color: white; padding: 15px 25px; border-radius: 8px; width: 180px; text-align: center; box-shadow: 2px 2px 5px rgba(0,0,0,0.1);">
                <div style="font-size: 14px; text-transform: uppercase; opacity: 0.8;">Total Units</div>
                <div style="font-size: 24px; font-weight: bold; margin-top: 5px;">{total_units:.1f}</div>
            </div>
        </div>
        """
        display(widgets.HTML(kpi_html))

        # --- Graphical Elements: Styled Table ---
        html_table = "<table style='width: 100%; border-collapse: collapse; margin-top: 15px; font-family: sans-serif; box-shadow: 0 4px 6px rgba(0,0,0,0.05);'>"
        html_table += "<tr style='background-color: #34495e; color: white; text-align: left; font-weight: bold;'>"
        html_table += "<th style='padding: 12px; border: 1px solid #ddd;'>Course Code</th>"
        html_table += "<th style='padding: 12px; border: 1px solid #ddd;'>Course Title</th>"
        html_table += "<th style='padding: 12px; border: 1px solid #ddd; text-align: center;'>Units</th>"
        html_table += "<th style='padding: 12px; border: 1px solid #ddd;'>Instructor</th>"
        html_table += "<th style='padding: 12px; border: 1px solid #ddd; text-align: center;'>Grade</th></tr>"

        for idx, row in filtered_df.iterrows():
            code = str(row.get('Course Code', '')).strip()
            title = str(row.get('Course Title', '')).strip()
            units = str(row.get('Units', '')).strip()
            inst = str(row.get('Instructor', '')).strip()
            grade = str(row.get(grade_col, '')).strip()

            # Alternate row background
            bg_color = "#f8f9f9" if idx % 2 == 0 else "#ffffff"

            html_table += f"<tr style='background-color: {bg_color}; border-bottom: 1px solid #ddd;'>"
            html_table += f"<td style='padding: 10px; border: 1px solid #ddd; font-weight: bold; color: #2c3e50;'>{code}</td>"
            html_table += f"<td style='padding: 10px; border: 1px solid #ddd;'>{title}</td>"
            html_table += f"<td style='padding: 10px; border: 1px solid #ddd; text-align: center;'>{units}</td>"
            html_table += f"<td style='padding: 10px; border: 1px solid #ddd; font-style: italic; color: #7f8c8d;'>{inst}</td>"
            html_table += f"<td style='padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; background-color: #e8f8f5; color: #16a085;'>{grade}</td></tr>"

        html_table += "</table>"
        display(widgets.HTML(html_table))

# Bind event handlers
upload_btn.on_click(on_upload_clicked)
view_btn.on_click(on_view_clicked)

# Arrange Layout in a premium styled container
controls = widgets.HBox([upload_btn, student_dropdown, view_btn], layout=widgets.Layout(align_items='center', justify_content='center', padding='10px', background_color='#f4f6f9', border_radius='8px'))
title = widgets.HTML("<div style='text-align: center; color: #2c3e50; font-family: sans-serif; padding-bottom: 15px;'>"
                     "<h2 style='margin: 0;'>🎓 Student Academic Record Dashboard</h2>"
                     "<p style='margin: 5px 0 0 0; color: #7f8c8d;'>Upload a CSV file to view consolidated cards and grades on one sheet</p>"
                     "</div>")

display(title, controls, output_area)

HTML(value="<div style='text-align: center; color: #2c3e50; font-family: sans-serif; padding-bottom: 15px;'><h…

Output()